# EMI Analysis: Mean-Centered Prototypes + Time-Series Trends

Two analyses:
1. **Mean-centered prototypes** — subtract the mean SAE feature vector computed over all 1M speeches (the activations already collected) from both prototype vectors before scoring. This removes the shared baseline activation across all congressional speech and isolates pole-specific signal.
2. **Time-series trends** — plot decade-level mean EMI for all three methods (W2V, BERT, GPT-2) to check whether they agree on the direction of change over 1880–2020.

**Why 1M and not Phase 2 (75K)?**  
The 1M raw activations are already on disk from the activation collection pipeline. Running them through the SAE encoder (tiny operation — no full BERT/GPT-2 needed) and accumulating a running mean gives a far more stable estimate of the *average congressional speech* in SAE feature space. We never materialise all 1M × d_sae features at once — just keep a running sum.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')   # non-interactive backend for HPC; change to 'TkAgg' locally
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

RESULTS_DIR   = '/content/drive/MyDrive/EMI_Project/results/sparse_sae_emi'
FIGURES_DIR   = '/content/drive/MyDrive/EMI_Project/results/sparse_sae_emi/figures'
ACTIVATIONS_DIR = '/content/drive/MyDrive/EMI_Project/activations'
os.makedirs(FIGURES_DIR, exist_ok=True)

N_EV          = 100_000   # must match PHASE1_TOP_N_PER_POLE
CHUNK_SIZE    = 100_000   # rows processed at once when computing 1M mean

# 1M activation files (from collect_activations_bert/gpt2.py)
BERT_1M_PATH  = f'{ACTIVATIONS_DIR}/bert_last_1000000.npy'
GPT2_1M_PATH  = f'{ACTIVATIONS_DIR}/gpt2_last_1000000.npy'

# SAE checkpoints — same auto-detection as the pipeline
import glob
def _auto_detect_sae(sae_dir, model_tag):
    files = sorted(glob.glob(f'{sae_dir}/sae_{model_tag}_*.pt'), key=os.path.getmtime)
    return files[-1] if files else None

SAE_BERT_PATH = _auto_detect_sae(f'{ACTIVATIONS_DIR}/sae/bert', 'bert')
SAE_GPT2_PATH = _auto_detect_sae(f'{ACTIVATIONS_DIR}/sae/gpt2', 'gpt2')

print(f'Device:        {device}')
print(f'BERT 1M acts:  {BERT_1M_PATH}  exists={os.path.isfile(BERT_1M_PATH)}')
print(f'GPT-2 1M acts: {GPT2_1M_PATH}  exists={os.path.isfile(GPT2_1M_PATH)}')
print(f'BERT SAE:      {SAE_BERT_PATH}')
print(f'GPT-2 SAE:     {SAE_GPT2_PATH}')

Device:        cpu
BERT 1M acts:  /content/drive/MyDrive/EMI_Project/activations/bert_last_1000000.npy  exists=True
GPT-2 1M acts: /content/drive/MyDrive/EMI_Project/activations/gpt2_last_1000000.npy  exists=True
BERT SAE:      /content/drive/MyDrive/EMI_Project/activations/sae/bert/sae_bert_exp8_l15e-04.pt
GPT-2 SAE:     /content/drive/MyDrive/EMI_Project/activations/sae/gpt2/sae_gpt2_exp4_l11e-03.pt


In [12]:
# ── SparseAutoencoder (must match train_sae.ipynb exactly) ────────────────────
class SparseAutoencoder(nn.Module):
    def __init__(self, d_in, d_sae):
        super().__init__()
        self.d_in  = d_in
        self.d_sae = d_sae
        self.W_enc = nn.Linear(d_in, d_sae, bias=True)
        self.W_dec = nn.Linear(d_sae, d_in, bias=True)
    def encode(self, x):
        return F.relu(self.W_enc(x))
    def forward(self, x):
        f = self.encode(x)
        return f, self.W_dec(f)

def load_sae(path):
    ckpt     = torch.load(path, map_location=device, weights_only=False)
    cfg      = ckpt['config']
    sae      = SparseAutoencoder(cfg['d_in'], cfg['d_sae']).to(device)
    sae.load_state_dict(ckpt['model_state_dict'])
    sae.eval()
    act_mean = np.array(cfg['act_mean'], dtype=np.float32)   # (1, d_in)
    print(f'  Loaded SAE  d_in={cfg["d_in"]}  d_sae={cfg["d_sae"]}  from {path}')
    return sae, act_mean

print('Loading SAEs...')
bert_sae, bert_act_mean = load_sae(SAE_BERT_PATH)
gpt2_sae, gpt2_act_mean = load_sae(SAE_GPT2_PATH)

Loading SAEs...
  Loaded SAE  d_in=768  d_sae=6144  from /content/drive/MyDrive/EMI_Project/activations/sae/bert/sae_bert_exp8_l15e-04.pt
  Loaded SAE  d_in=1024  d_sae=4096  from /content/drive/MyDrive/EMI_Project/activations/sae/gpt2/sae_gpt2_exp4_l11e-03.pt


In [13]:
# ── Load pre-computed artefacts ────────────────────────────────────────────────
bert_ev_proto    = np.load(f'{RESULTS_DIR}/bert_evidence_prototype.npy')
bert_in_proto    = np.load(f'{RESULTS_DIR}/bert_intuition_prototype.npy')
gpt2_ev_proto    = np.load(f'{RESULTS_DIR}/gpt2_evidence_prototype.npy')
gpt2_in_proto    = np.load(f'{RESULTS_DIR}/gpt2_intuition_prototype.npy')

print('Loading Phase 2 SAE features...')
bert_p2_feats    = np.load(f'{RESULTS_DIR}/bert_phase2_sae_features.npy')
gpt2_p2_feats    = np.load(f'{RESULTS_DIR}/gpt2_phase2_sae_features.npy')

bert_p2_emi_orig = np.load(f'{RESULTS_DIR}/bert_phase2_emi_scores.npy')
gpt2_p2_emi_orig = np.load(f'{RESULTS_DIR}/gpt2_phase2_emi_scores.npy')

df_orig          = pd.read_csv(f'{RESULTS_DIR}/emi_comparison_phase2.csv')
w2v_emi          = df_orig['w2v_emi'].values

print(f'BERT prototypes:  d_sae={len(bert_ev_proto)}')
print(f'GPT-2 prototypes: d_sae={len(gpt2_ev_proto)}')
print(f'Phase 2 CSV:      {df_orig.shape}')

Loading Phase 2 SAE features...
BERT prototypes:  d_sae=6144
GPT-2 prototypes: d_sae=4096
Phase 2 CSV:      (75000, 8)


## Section 1 — Compute 1M-Speech SAE Feature Mean

Stream through the 1M raw activations in chunks of 100K, encode each chunk through the SAE, and accumulate a running sum. The mean is computed at the end. Peak memory = one chunk of activations + one chunk of features (a few hundred MB at most).

In [14]:
BERT_1M_MEAN_PATH = f'{RESULTS_DIR}/bert_1m_sae_mean.npy'
GPT2_1M_MEAN_PATH = f'{RESULTS_DIR}/gpt2_1m_sae_mean.npy'

def compute_1m_sae_mean(acts_path, sae, act_mean, chunk_size, label):
    """
    Load activations with mmap (no full copy into RAM), encode in chunks,
    return mean feature vector shape (d_sae,).
    """
    acts = np.load(acts_path, mmap_mode='r')   # memory-mapped, ~3-4 GB on disk
    n    = len(acts)
    print(f'{label}: {n:,} activations  shape={acts.shape}')

    running_sum = np.zeros(sae.d_sae, dtype=np.float64)

    sae.eval()
    with torch.no_grad():
        for start in range(0, n, chunk_size):
            end   = min(start + chunk_size, n)
            chunk = acts[start:end].astype(np.float32) - act_mean   # centre by SAE act_mean
            feats = sae.encode(torch.from_numpy(chunk).to(device)).float().cpu().numpy()
            running_sum += feats.sum(axis=0).astype(np.float64)
            if (start // chunk_size) % 2 == 0:
                print(f'  {end:,}/{n:,}  chunk sparsity: {(feats > 0).mean()*100:.1f}% active')

    mean_vec = (running_sum / n).astype(np.float32)
    print(f'{label} mean: {(mean_vec > 0).sum():,} positive features / {sae.d_sae:,}')
    return mean_vec


if os.path.isfile(BERT_1M_MEAN_PATH):
    bert_1m_mean = np.load(BERT_1M_MEAN_PATH)
    print(f'Loaded cached BERT 1M mean: {bert_1m_mean.shape}')
else:
    print('Computing BERT 1M SAE mean...')
    bert_1m_mean = compute_1m_sae_mean(BERT_1M_PATH, bert_sae, bert_act_mean, CHUNK_SIZE, 'BERT')
    np.save(BERT_1M_MEAN_PATH, bert_1m_mean)
    print(f'Saved: {BERT_1M_MEAN_PATH}')

if os.path.isfile(GPT2_1M_MEAN_PATH):
    gpt2_1m_mean = np.load(GPT2_1M_MEAN_PATH)
    print(f'Loaded cached GPT-2 1M mean: {gpt2_1m_mean.shape}')
else:
    print('Computing GPT-2 1M SAE mean...')
    gpt2_1m_mean = compute_1m_sae_mean(GPT2_1M_PATH, gpt2_sae, gpt2_act_mean, CHUNK_SIZE, 'GPT-2')
    np.save(GPT2_1M_MEAN_PATH, gpt2_1m_mean)
    print(f'Saved: {GPT2_1M_MEAN_PATH}')

Loaded cached BERT 1M mean: (6144,)
Loaded cached GPT-2 1M mean: (4096,)


## Section 2 — Mean-Centered Prototype Analysis

In [15]:
# ── Build centered prototypes ─────────────────────────────────────────────────
bert_ev_centered = bert_ev_proto - bert_1m_mean
bert_in_centered = bert_in_proto - bert_1m_mean
gpt2_ev_centered = gpt2_ev_proto - gpt2_1m_mean
gpt2_in_centered = gpt2_in_proto - gpt2_1m_mean

def proto_cosine(a, b):
    return float((a / np.linalg.norm(a)) @ (b / np.linalg.norm(b)))

orig_bert_cos     = proto_cosine(bert_ev_proto,    bert_in_proto)
orig_gpt2_cos     = proto_cosine(gpt2_ev_proto,    gpt2_in_proto)
centered_bert_cos = proto_cosine(bert_ev_centered, bert_in_centered)
centered_gpt2_cos = proto_cosine(gpt2_ev_centered, gpt2_in_centered)

print('Prototype cosine similarity (lower = better separation):')
print(f'  BERT  original:  {orig_bert_cos:+.4f}')
print(f'  BERT  centered:  {centered_bert_cos:+.4f}   Δ = {centered_bert_cos - orig_bert_cos:+.4f}')
print(f'  GPT-2 original:  {orig_gpt2_cos:+.4f}')
print(f'  GPT-2 centered:  {centered_gpt2_cos:+.4f}   Δ = {centered_gpt2_cos - orig_gpt2_cos:+.4f}')

print()
print('How many prototype features flip sign after centering?')
for label, ev_p, in_p, ev_c, in_c in [
    ('BERT',  bert_ev_proto, bert_in_proto, bert_ev_centered, bert_in_centered),
    ('GPT-2', gpt2_ev_proto, gpt2_in_proto, gpt2_ev_centered, gpt2_in_centered),
]:
    ev_flip = ((ev_p > 0) & (ev_c < 0)).sum()
    in_flip = ((in_p > 0) & (in_c < 0)).sum()
    d       = len(ev_p)
    print(f'  {label}  evidence: {ev_flip:,}/{d:,} ({ev_flip/d*100:.1f}%) flipped   '
          f'intuition: {in_flip:,}/{d:,} ({in_flip/d*100:.1f}%) flipped')

Prototype cosine similarity (lower = better separation):
  BERT  original:  +0.9506
  BERT  centered:  -0.6397   Δ = -1.5903
  GPT-2 original:  +0.9134
  GPT-2 centered:  -0.1321   Δ = -1.0455

How many prototype features flip sign after centering?
  BERT  evidence: 1,483/6,144 (24.1%) flipped   intuition: 533/6,144 (8.7%) flipped
  GPT-2  evidence: 3,246/4,096 (79.2%) flipped   intuition: 790/4,096 (19.3%) flipped


In [16]:
# ── Recompute Phase 2 EMI with 1M-centered features and prototypes ────────────

def compute_emi_cosine_centered(feats, ev_centered, in_centered, global_mean):
    feats_c    = feats.astype(np.float32) - global_mean
    ev_n       = ev_centered / np.linalg.norm(ev_centered)
    in_n       = in_centered / np.linalg.norm(in_centered)
    norms      = np.linalg.norm(feats_c, axis=1, keepdims=True)
    norms      = np.where(norms == 0, 1.0, norms)
    feats_unit = feats_c / norms
    return (feats_unit @ ev_n - feats_unit @ in_n).astype(np.float32)

bert_p2_emi_centered = compute_emi_cosine_centered(
    bert_p2_feats, bert_ev_centered, bert_in_centered, bert_1m_mean
)
gpt2_p2_emi_centered = compute_emi_cosine_centered(
    gpt2_p2_feats, gpt2_ev_centered, gpt2_in_centered, gpt2_1m_mean
)

def compare_methods(orig, centered, w2v, label):
    r_orig, _ = pearsonr(w2v, orig)
    r_cent, _ = pearsonr(w2v, centered)
    thr       = np.percentile(w2v, 75)
    binary    = (w2v > thr).astype(int)
    auc_orig  = roc_auc_score(binary, orig)
    auc_cent  = roc_auc_score(binary, centered)
    print(f'{label}')
    print(f'  Pearson r vs W2V:  original={r_orig:+.4f}   centered={r_cent:+.4f}   '
          f'Δ={r_cent-r_orig:+.4f}')
    print(f'  AUC top-quartile:  original={auc_orig:.4f}   centered={auc_cent:.4f}   '
          f'Δ={auc_cent-auc_orig:+.4f}')

compare_methods(bert_p2_emi_orig, bert_p2_emi_centered, w2v_emi, 'BERT')
compare_methods(gpt2_p2_emi_orig, gpt2_p2_emi_centered, w2v_emi, 'GPT-2')

BERT
  Pearson r vs W2V:  original=+0.5318   centered=+0.5385   Δ=+0.0067
  AUC top-quartile:  original=0.7915   centered=0.7842   Δ=-0.0073
GPT-2
  Pearson r vs W2V:  original=+0.3358   centered=+0.3320   Δ=-0.0038
  AUC top-quartile:  original=0.6831   centered=0.6784   Δ=-0.0047


In [17]:
# ── Top features of centered prototypes ──────────────────────────────────────
def top_centered_features(ev_centered, in_centered, top_k=10, label=''):
    diff = ev_centered - in_centered
    print(f'\n{label} — top evidence features (after centering):')
    for idx in np.argsort(diff)[::-1][:top_k]:
        print(f'  feat {idx:5d}  diff={diff[idx]:+.4f}  '
              f'ev={ev_centered[idx]:+.4f}  in={in_centered[idx]:+.4f}')
    print(f'{label} — top intuition features (after centering):')
    for idx in np.argsort(diff)[:top_k]:
        print(f'  feat {idx:5d}  diff={diff[idx]:+.4f}  '
              f'ev={ev_centered[idx]:+.4f}  in={in_centered[idx]:+.4f}')

top_centered_features(bert_ev_centered, bert_in_centered, label='BERT')
top_centered_features(gpt2_ev_centered, gpt2_in_centered, label='GPT-2')


BERT — top evidence features (after centering):
  feat   440  diff=+1.8059  ev=+1.6649  in=-0.1410
  feat   783  diff=+1.0029  ev=+0.9647  in=-0.0382
  feat   796  diff=+0.5751  ev=+0.5581  in=-0.0171
  feat  5133  diff=+0.4579  ev=+0.4866  in=+0.0287
  feat  4237  diff=+0.2525  ev=+0.1799  in=-0.0726
  feat  3745  diff=+0.2475  ev=+0.2340  in=-0.0135
  feat  6001  diff=+0.2152  ev=+0.1811  in=-0.0340
  feat  1736  diff=+0.1957  ev=+0.1363  in=-0.0594
  feat  5537  diff=+0.1804  ev=+0.1734  in=-0.0070
  feat  4758  diff=+0.1758  ev=+0.1350  in=-0.0407
BERT — top intuition features (after centering):
  feat  4161  diff=-1.4064  ev=-0.9976  in=+0.4088
  feat  5465  diff=-1.2188  ev=-0.7147  in=+0.5041
  feat  4432  diff=-0.7059  ev=-0.5261  in=+0.1798
  feat  5454  diff=-0.6136  ev=-0.4229  in=+0.1907
  feat  2947  diff=-0.6060  ev=-0.4335  in=+0.1725
  feat  5267  diff=-0.5283  ev=-0.3974  in=+0.1309
  feat  4345  diff=-0.5084  ev=-0.3831  in=+0.1253
  feat  3865  diff=-0.4055  ev=-0.3

## Section 3 — Time-Series Trends by Decade

In [18]:
# ── Build comparison DataFrame ────────────────────────────────────────────────
df = df_orig.copy()
df['bert_emi_centered'] = bert_p2_emi_centered
df['gpt2_emi_centered'] = gpt2_p2_emi_centered

for col in ['w2v_emi', 'bert_emi', 'gpt2_emi', 'bert_emi_centered', 'gpt2_emi_centered']:
    df[f'{col}_z'] = (df[col] - df[col].mean()) / df[col].std()

decade_stats = df.groupby('decade').agg(
    n           = ('w2v_emi', 'size'),
    w2v_mean    = ('w2v_emi_z',            'mean'),
    w2v_se      = ('w2v_emi_z',            lambda x: x.std() / np.sqrt(len(x))),
    bert_mean   = ('bert_emi_z',           'mean'),
    bert_se     = ('bert_emi_z',           lambda x: x.std() / np.sqrt(len(x))),
    gpt2_mean   = ('gpt2_emi_z',           'mean'),
    gpt2_se     = ('gpt2_emi_z',           lambda x: x.std() / np.sqrt(len(x))),
    bert_c_mean = ('bert_emi_centered_z',  'mean'),
    bert_c_se   = ('bert_emi_centered_z',  lambda x: x.std() / np.sqrt(len(x))),
    gpt2_c_mean = ('gpt2_emi_centered_z',  'mean'),
    gpt2_c_se   = ('gpt2_emi_centered_z',  lambda x: x.std() / np.sqrt(len(x))),
    w2v_raw     = ('w2v_emi',              'mean'),
    bert_raw    = ('bert_emi',             'mean'),
    gpt2_raw    = ('gpt2_emi',             'mean'),
    bert_c_raw  = ('bert_emi_centered',    'mean'),
    gpt2_c_raw  = ('gpt2_emi_centered',    'mean'),
).reset_index()

print(decade_stats[['decade','n','w2v_mean','bert_mean','gpt2_mean',
                     'bert_c_mean','gpt2_c_mean']].to_string(index=False))

 decade    n  w2v_mean  bert_mean  gpt2_mean  bert_c_mean  gpt2_c_mean
   1880 5000  0.077650  -0.065802   0.045113     0.097336     0.004801
   1890 5000  0.013271  -0.133828  -0.123089     0.032648    -0.099300
   1900 5000 -0.017842  -0.221367  -0.309574    -0.070203    -0.230691
   1910 5000 -0.055042  -0.245947  -0.327506    -0.094617    -0.275202
   1920 5000 -0.068547  -0.262777  -0.467272    -0.124203    -0.390126
   1930 5000 -0.057784  -0.228913  -0.716418    -0.149679    -0.514635
   1940 5000 -0.033231  -0.178601  -0.723845    -0.156743    -0.426914
   1950 5000  0.026035  -0.154425  -0.671777    -0.148735    -0.324700
   1960 5000  0.077910  -0.112578  -0.419106    -0.179508    -0.079189
   1970 5000  0.125816   0.004163  -0.042186    -0.082598     0.446538
   1980 5000  0.093267   0.130257   0.535886     0.041779     0.834794
   1990 5000 -0.034612   0.169241   0.756119     0.016565     0.757209
   2000 5000  0.001284   0.325838   0.941301     0.136953     0.467265
   201

In [19]:
# ── Plot 1: All three methods, z-scored ───────────────────────────────────────
decades = decade_stats['decade'].values
COLOURS = {'w2v': '#2171b5', 'bert': '#e6550d', 'gpt2': '#31a354'}

fig, ax = plt.subplots(figsize=(12, 5))
for col, label, colour in [
    ('w2v',  'Word2Vec', COLOURS['w2v']),
    ('bert', 'BERT',     COLOURS['bert']),
    ('gpt2', 'GPT-2',    COLOURS['gpt2']),
]:
    mean = decade_stats[f'{col}_mean'].values
    se   = decade_stats[f'{col}_se'].values
    ax.plot(decades, mean, marker='o', markersize=5, label=label, color=colour, linewidth=2)
    ax.fill_between(decades, mean - se, mean + se, alpha=0.15, color=colour)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
ax.set_xlabel('Decade', fontsize=12)
ax.set_ylabel('Mean EMI (z-scored)', fontsize=12)
ax.set_title('Evidence-Motivation Index over Time\n(z-scored; ribbon = ±1 SE)', fontsize=13)
ax.legend(fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/emi_trends_zscore.png', dpi=150)
plt.show()
print(f'Saved: {FIGURES_DIR}/emi_trends_zscore.png')

Saved: /content/drive/MyDrive/EMI_Project/results/sparse_sae_emi/figures/emi_trends_zscore.png


In [20]:
# ── Plot 2: Original vs 1M-centered, BERT and GPT-2 side-by-side ─────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, col_orig, col_c, label, colour in [
    (axes[0], 'bert_mean', 'bert_c_mean', 'BERT',  COLOURS['bert']),
    (axes[1], 'gpt2_mean', 'gpt2_c_mean', 'GPT-2', COLOURS['gpt2']),
]:
    mean_o = decade_stats[col_orig].values
    se_o   = decade_stats[col_orig.replace('mean','se')].values
    mean_c = decade_stats[col_c].values
    se_c   = decade_stats[col_c.replace('mean','se')].values

    ax.plot(decades, mean_o, marker='o', markersize=4, label='Original',
            color=colour, linewidth=2)
    ax.fill_between(decades, mean_o - se_o, mean_o + se_o, alpha=0.15, color=colour)

    ax.plot(decades, mean_c, marker='s', markersize=4, label='1M-centered',
            color=colour, linewidth=2, linestyle='--')
    ax.fill_between(decades, mean_c - se_c, mean_c + se_c, alpha=0.1, color=colour)

    ax.plot(decades, decade_stats['w2v_mean'].values,
            color=COLOURS['w2v'], linewidth=1.5, linestyle=':', label='W2V', alpha=0.7)

    ax.axhline(0, color='black', linewidth=0.7, linestyle='--', alpha=0.4)
    ax.set_title(f'{label}: original vs 1M-centered', fontsize=12)
    ax.set_xlabel('Decade')
    ax.set_ylabel('Mean EMI (z-scored)' if ax is axes[0] else '')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Original vs 1M-Centered EMI by Decade (z-scored)', fontsize=13)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/emi_centered_comparison.png', dpi=150)
plt.show()
print(f'Saved: {FIGURES_DIR}/emi_centered_comparison.png')

Saved: /content/drive/MyDrive/EMI_Project/results/sparse_sae_emi/figures/emi_centered_comparison.png


In [21]:
# ── Plot 3: Raw bar charts — each method on its own scale ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
for ax, col, label, colour in [
    (axes[0], 'w2v_raw',   'Word2Vec', COLOURS['w2v']),
    (axes[1], 'bert_raw',  'BERT',     COLOURS['bert']),
    (axes[2], 'gpt2_raw',  'GPT-2',    COLOURS['gpt2']),
]:
    ax.bar(decades, decade_stats[col].values, width=7, color=colour, alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(label, fontsize=12)
    ax.set_xlabel('Decade')
    ax.set_ylabel('Mean EMI (raw)' if ax is axes[0] else '')
    ax.xaxis.set_major_locator(ticker.MultipleLocator(40))
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Raw Mean EMI per Decade (each method on its own scale)', fontsize=13)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/emi_raw_by_decade.png', dpi=150)
plt.show()
print(f'Saved: {FIGURES_DIR}/emi_raw_by_decade.png')

Saved: /content/drive/MyDrive/EMI_Project/results/sparse_sae_emi/figures/emi_raw_by_decade.png


In [22]:
# ── Plot 4: BERT EMI by party and decade ──────────────────────────────────────
df_party       = df[df['party'].isin(['D', 'R'])].copy()
party_decade   = df_party.groupby(['decade', 'party'])['bert_emi_z'].agg(
    mean='mean', se=lambda x: x.std() / np.sqrt(len(x))
).reset_index()
party_colours  = {'D': '#2171b5', 'R': '#cb181d'}

fig, ax = plt.subplots(figsize=(12, 5))
for party, grp in party_decade.groupby('party'):
    grp = grp.sort_values('decade')
    ax.plot(grp['decade'], grp['mean'], marker='o', markersize=4,
            label='Democrat' if party == 'D' else 'Republican',
            color=party_colours[party], linewidth=2)
    ax.fill_between(grp['decade'], grp['mean'] - grp['se'],
                    grp['mean'] + grp['se'], alpha=0.15, color=party_colours[party])

ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
ax.set_xlabel('Decade', fontsize=12)
ax.set_ylabel('Mean BERT EMI (z-scored)', fontsize=12)
ax.set_title('BERT EMI by Party and Decade', fontsize=13)
ax.legend(fontsize=11)
ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/emi_by_party.png', dpi=150)
plt.show()
print(f'Saved: {FIGURES_DIR}/emi_by_party.png')

/tmp/ipykernel_6077/650092860.py:21: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=11)


Saved: /content/drive/MyDrive/EMI_Project/results/sparse_sae_emi/figures/emi_by_party.png


In [23]:
# ── Numeric summary and decade-level correlations ────────────────────────────
print('Decade-level mean EMI (z-scored):')
print(f'{"Decade":>7}  {"W2V":>8}  {"BERT":>8}  {"GPT-2":>8}  {"BERT-C":>8}  {"GPT2-C":>8}')
for _, row in decade_stats.iterrows():
    print(f'{int(row.decade):>7}  '
          f'{row.w2v_mean:>+8.4f}  '
          f'{row.bert_mean:>+8.4f}  '
          f'{row.gpt2_mean:>+8.4f}  '
          f'{row.bert_c_mean:>+8.4f}  '
          f'{row.gpt2_c_mean:>+8.4f}')

print('\nPearson r of decade-level means:')
dec = decade_stats
for a_name, b_name, a, b in [
    ('W2V',    'BERT',     dec.w2v_mean,    dec.bert_mean),
    ('W2V',    'GPT-2',    dec.w2v_mean,    dec.gpt2_mean),
    ('W2V',    'BERT-C',   dec.w2v_mean,    dec.bert_c_mean),
    ('W2V',    'GPT2-C',   dec.w2v_mean,    dec.gpt2_c_mean),
    ('BERT',   'GPT-2',    dec.bert_mean,   dec.gpt2_mean),
    ('BERT',   'BERT-C',   dec.bert_mean,   dec.bert_c_mean),
    ('GPT-2',  'GPT2-C',   dec.gpt2_mean,   dec.gpt2_c_mean),
]:
    r, p = pearsonr(a, b)
    print(f'  {a_name:<8} vs {b_name:<8}  r={r:+.4f}  p={p:.3f}')

Decade-level mean EMI (z-scored):
 Decade       W2V      BERT     GPT-2    BERT-C    GPT2-C
   1880   +0.0776   -0.0658   +0.0451   +0.0973   +0.0048
   1890   +0.0133   -0.1338   -0.1231   +0.0326   -0.0993
   1900   -0.0178   -0.2214   -0.3096   -0.0702   -0.2307
   1910   -0.0550   -0.2459   -0.3275   -0.0946   -0.2752
   1920   -0.0685   -0.2628   -0.4673   -0.1242   -0.3901
   1930   -0.0578   -0.2289   -0.7164   -0.1497   -0.5146
   1940   -0.0332   -0.1786   -0.7238   -0.1567   -0.4269
   1950   +0.0260   -0.1544   -0.6718   -0.1487   -0.3247
   1960   +0.0779   -0.1126   -0.4191   -0.1795   -0.0792
   1970   +0.1258   +0.0042   -0.0422   -0.0826   +0.4465
   1980   +0.0933   +0.1303   +0.5359   +0.0418   +0.8348
   1990   -0.0346   +0.1692   +0.7561   +0.0166   +0.7572
   2000   +0.0013   +0.3258   +0.9413   +0.1370   +0.4673
   2010   -0.0947   +0.4093   +0.8073   +0.2252   +0.0450
   2020   -0.0534   +0.5655   +0.7151   +0.4558   -0.2148

Pearson r of decade-level means:
  W2

In [24]:
# ── Save outputs ──────────────────────────────────────────────────────────────
np.save(f'{RESULTS_DIR}/bert_phase2_emi_centered.npy', bert_p2_emi_centered)
np.save(f'{RESULTS_DIR}/gpt2_phase2_emi_centered.npy', gpt2_p2_emi_centered)

df.to_csv(f'{RESULTS_DIR}/emi_comparison_phase2_with_centered.csv', index=False)

print('Outputs saved:')
for fname in [
    'bert_1m_sae_mean.npy', 'gpt2_1m_sae_mean.npy',
    'bert_phase2_emi_centered.npy', 'gpt2_phase2_emi_centered.npy',
    'emi_comparison_phase2_with_centered.csv',
]:
    path = f'{RESULTS_DIR}/{fname}'
    if os.path.isfile(path):
        print(f'  {fname:<52} ({os.path.getsize(path)/1e6:.1f} MB)')

print(f'\nFigures in: {FIGURES_DIR}')
for fname in os.listdir(FIGURES_DIR):
    print(f'  {fname}')

Outputs saved:
  bert_1m_sae_mean.npy                                 (0.0 MB)
  gpt2_1m_sae_mean.npy                                 (0.0 MB)
  bert_phase2_emi_centered.npy                         (0.3 MB)
  gpt2_phase2_emi_centered.npy                         (0.3 MB)
  emi_comparison_phase2_with_centered.csv              (13.2 MB)

Figures in: /content/drive/MyDrive/EMI_Project/results/sparse_sae_emi/figures
  emi_trends_zscore.png
  emi_centered_comparison.png
  emi_raw_by_decade.png
  emi_by_party.png
